# Time series Notebook
The aim of this notebook is to compare the timeseries of topic inside/between clusters.

## Load libraries

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [18]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
from bertopic import BERTopic
import polars as pl

## Load data and create the timeseries dataset

In [3]:
cluster_df = pd.read_parquet('community_detection_results_4B_0_0_5.parquet')

In [4]:
cluster_df['Cluster Label'].unique()

array(['Multidrug-resistant TB Research', 'Zoonotic Disease Surveillance',
       'Mosquito-borne Diseases', 'Food Safety and Pathogen Risk',
       'Antibiotic Resistance Dynamics',
       'Influenza Viruses and Pandemics', 'Monkeypox Outbreak Response',
       'Bovine TB and Culling Controversy', 'SARS-CoVs Variant Dynamics',
       'Infection Control Practices', 'Vaccine Safety Monitoring',
       'Air Quality Policy Impact', 'Malaria Prevention Strategies',
       'Pandemic Response Strategies', 'Zika and Arbovirus Research',
       'Pandemic Impact on Education and Health',
       'HIV and Related Viral Dynamics', 'Polio Prevention Strategies',
       'Sexually Transmitted Infections',
       'Vaccine Immunity and Outbreaks',
       'Antiviral Therapies for COVID-19', 'Zoonotic Virus Emergences',
       'HIV Care and Epidemic Management', 'Vaccine Information Dynamics',
       'Maternal and Child Health', 'Antiviral Drug Resistance Policies',
       'Diagnostic Technologies and Te

In [5]:
model_list = ['scopus','science_news','the_guardian']

In [6]:
timeseries_df = pd.DataFrame()

In [7]:
cfg_dict = cfg.MAGAZINE_CONFIG['the_guardian']
model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

In [8]:
pd.DataFrame({'id': model_data['id'],
             'summary':model_data['text']}).to_csv('summary.csv',sep='ç',encoding='utf-8')

In [9]:
for model in model_list:
    
    cfg_dict = cfg.MAGAZINE_CONFIG[model]
    bertopic_model = BERTopic.load(cfg_dict['REFERENCE_MODEL'])

    model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

    dataset = pd.read_parquet(cfg_dict['DATASET_PATH'])
    
    tmp_df = bertopic_model.get_document_info(model_data['text'])['CustomName'].reset_index()
    tmp_df = tmp_df.drop(columns='index')

    tmp_df['id'] =  list(model_data['id'])
    tmp_df = tmp_df.rename(columns={'CustomName':'Topic Label'})
    
    tmp_df = tmp_df.merge(cluster_df,on='Topic Label')

    columns_to_mantain = ['id','publicationDate']
    columns_to_drop = [column for column in dataset.columns if column not in columns_to_mantain ]

    tmp_df = tmp_df.merge(dataset.drop(columns=columns_to_drop),on='id')

    timeseries_df = pd.concat([timeseries_df,tmp_df])


2026-05-29 16:06:04,817 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-05-29 16:06:18,230 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-05-29 16:06:19,657 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


### Show clusters and topics

In [10]:
cluster_df[['Cluster','Cluster Label']].value_counts().reset_index().sort_values(by='Cluster') 

,Cluster,Cluster Label,count
0,1,Pandemic Response Strategies,17
1,2,Antibiotic Resistance Dynamics,12
2,3,Influenza Viruses and Pandemics,10
3,4,Zoonotic Virus Emergences,10
10,5,Zoonotic Disease Surveillance,7
6,6,Pandemic Impact on Education and Health,8
13,7,Mosquito-borne Diseases,6
4,9,Vaccine Immunity and Outbreaks,10
8,10,Microbial and Genetic Therapies,7
15,11,HIV Care and Epidemic Management,5


## New graph - Splitted per model and normalized per model/year

### Load the dataset that we use to normalize 

In [11]:
model_articles_per_year = pd.read_parquet('model_articles_per_year.parquet')

### Create the timeseries datasets

Choose which clusters show

In [12]:
#cluster_list = list(range(0,10))
#cluster_list = list(range(10,20))
#cluster_list = list(range(20,30))
#cluster_list = list(range(30,45))
#Covid
#cluster_list = [6,15,16,19,20,42,40,22,34]
#vaccine
#cluster_list = [9,27,32,45,46,47,48]
# school
#cluster_list = [6]
cluster_list = [3]

In [13]:
timeseries_df_analysis = timeseries_df[ (timeseries_df['Cluster'].isin(cluster_list)) & (timeseries_df['publicationDate'].dt.year <= 2025)  ]

In [14]:
timeseries_df_analysis['Model'] = timeseries_df_analysis['Model'].apply( lambda x: x.title().replace('_',' '))

Set graph labels

In [15]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in timeseries_df_analysis.columns:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster Label']
    else:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']

Choose the granularity of time period

In [16]:
period = '1y'
#period = '1mo'

Truncate date based on the time period

In [19]:
timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis).with_columns(
    pl.col('publicationDate').dt.truncate(period).alias('timestamp'),
)

In [20]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','Model','timestamp']).agg(
        pl.len().alias('occurrence')
    ).sort(by=['timestamp','Model','Graph_label'],descending=False)

In [21]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= period,
    eager = True
)

In [22]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(['Graph_label','Model']).unique()
)

In [23]:
grid = (
    topics.join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [24]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [25]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','Model','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Model','Graph_label'],descending=False)
)

In [26]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

### Params to decide the time granularity that will be displayed on graph

In [27]:
if period == '1y':
    how = 'frequency_per_year'
    complete_timeseries_df_analysis = complete_timeseries_df_analysis.merge(model_articles_per_year,on=['Model','timestamp'])
    complete_timeseries_df_analysis['frequency_per_year'] = (complete_timeseries_df_analysis['occurrence'] / complete_timeseries_df_analysis['Articles']) * 100_000
else:
    how = 'occurrence'

### Graph plot

In [29]:
import plotly.express as px
import pandas as pd

models  = list(pd.unique(complete_timeseries_df_analysis["Model"]))

if len(cluster_list) > 1:
    legend_title = "Cluster"
    graph_title = "Cluster time series"
else:
    legend_title = "Topics"
    graph_title= "Topics time series"

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y=how,
    color="Graph_label",
    markers=True,
    line_group="Graph_label",
    facet_row="Model",
    title=graph_title,
    facet_row_spacing=0.12
)


fig.update_layout(
    title={
        'text': graph_title,
        'x': 0.5,
        'y' : 0.98,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    legend=dict(
        title=legend_title,
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    margin=dict(l=80, r=80, t=80, b=120),
    hovermode='x unified'
)


for r in range(1, len(models) + 1):
    fig.update_xaxes(
        title_text="Time" if r == 1 else None,  
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
        showticklabels=True,   
        ticks="outside"
    )

for r in range(1, len(models) + 1):
    fig.update_yaxes(
        title_text="# Articles" if r == 2 and how=='occurrence' else '# Articles per 100k' if r == 2 and how=='frequency_per_year' else  None,
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
    )

for ann in fig.layout.annotations:
    if ann.text.startswith("Model="):
        ann.text = ann.text.replace("Model=", "") 
        ann.x = 0.5                                
        ann.xanchor = "center"
        ann.y += 0.15                              
        ann.yanchor = "bottom"
        ann.font = dict(size=15, family="Arial, sans-serif")
        ann.textangle = 0 


fig.update_yaxes(matches=None)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()


In [30]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import pandas as pd

models = list(pd.unique(complete_timeseries_df_analysis["Model"]))
labels = list(pd.unique(complete_timeseries_df_analysis["Graph_label"]))

y_axis_labels = []

for r in range(1, len(models) + 1):
    if r == 2:
        y_axis_labels.append(
            "<b># Articles</b>"
            if how == "occurrence"
            else "<b># Articles per 100k</b>"
            if how == "frequency_per_year"
            else f"<b>{how}</b>"
        )
    else: 
        y_axis_labels.append('')

y_axis_labels.append("<b># Articles</b>")

if len(cluster_list) > 1:
    legend_title = "Clusters"
    hist_subplot_title = 'cluster'
    graph_title = "Clusters time series"
else:
    legend_title = "Topics"
    hist_subplot_title = 'topic'
    graph_title = "Topics time series"


# Colori coerenti tra lineplot e istogramma
palette = px.colors.qualitative.Plotly
color_map = {
    label: palette[i % len(palette)]
    for i, label in enumerate(labels)
}


# Una riga per ogni Model + una riga finale per l'istogramma comune
n_rows = len(models) + 1

row_heights = [0.75 / len(models)] * len(models) + [0.25]

subplot_titles = ["<b>"+str(model)+"</b>" for model in models] + [f"<b>Overall {hist_subplot_title} volume over time</b>"]


fig = make_subplots(
    rows=n_rows,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.15,
    row_heights=row_heights,
    subplot_titles=subplot_titles
)


shown_legend = set()

# -------------------------
# 1. Lineplot per ogni Model
# -------------------------
for i, model in enumerate(models):
    df_model = complete_timeseries_df_analysis[
        complete_timeseries_df_analysis["Model"] == model
    ]

    row = i + 1

    for label in labels:
        df_label = df_model[df_model["Graph_label"] == label]

        if df_label.empty:
            continue

        show_legend = label not in shown_legend

        fig.add_trace(
            go.Scatter(
                x=df_label["timestamp"],
                y=df_label[how],
                mode="lines+markers",
                name=str(label),
                legendgroup=str(label),
                showlegend=show_legend,
                line=dict(
                    width=2.5,
                    color=color_map[label]
                ),
                marker=dict(size=6),
                hovertemplate=(
                    "Time=%{x}<br>"
                    f"{how}=%{{y}}<br>"
                    f"{legend_title}={label}<extra></extra>"
                )
            ),
            row=row,
            col=1
        )

        shown_legend.add(label)


# --------------------------------------
# 2. Istogramma stacked comune ai modelli
# --------------------------------------

# Se vuoi un istogramma comune, devi aggregare ignorando Model.
# Qui sommo occurrence per timestamp e Graph_label.
hist_df = (
    complete_timeseries_df_analysis
    .groupby(["timestamp", "Graph_label"], as_index=False)["occurrence"]
    .sum()
)

hist_row = n_rows

for label in labels:
    df_label = hist_df[hist_df["Graph_label"] == label]

    if df_label.empty:
        continue

    fig.add_trace(
        go.Bar(
            x=df_label["timestamp"],
            y=df_label["occurrence"],
            name=str(label),
            legendgroup=str(label),
            showlegend=False,
            marker_color=color_map[label],
            opacity=0.85,
            hovertemplate=(
                "Time=%{x}<br>"
                "Articles=%{y}<br>"
                f"{legend_title}={label}<extra></extra>"
            )
        ),
        row=hist_row,
        col=1
    )


fig.update_layout(
    title={
        "text": "<b>"+graph_title+"</b>",
        "x": 0.5,
        "y": 0.98,
        "xanchor": "center",
        "font": {"size": 20, "family": "Arial, sans-serif"}
    },
    legend=dict(
        title=legend_title,
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    width=1250,
    height=850,
    margin=dict(l=80, r=80, t=100, b=140),
    hovermode="x unified",
    barmode="stack"
)


# -------------------------
# Formattazione assi X e Y
# -------------------------
for r in range(1, n_rows + 1):
    fig.update_xaxes(
        title_text="<b>Time</b>" if r == n_rows else None,
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgray",
        showline=True,
        linewidth=2,
        linecolor="black",
        showticklabels=True,
        ticks="outside"
    )

    fig.update_yaxes(
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgray",
        showline=True,
        linewidth=2,
        linecolor="black",
        title_standoff=40,
        ticklabelstandoff=8,
        automargin=False,
        #ticklabelposition="inside",
    )


fig.update_yaxes(
        title_text=None,
        row=2,
        col=1
)



for r, label in enumerate(y_axis_labels, start=1):
    axis_name = "yaxis" if r == 1 else f"yaxis{r}"
    y_domain = fig.layout[axis_name].domain
    y_mid = sum(y_domain) / 2

    fig.add_annotation(
        text=label,
        xref="paper",
        yref="paper",
        x=-0.08,
        y=y_mid,
        textangle=-90,
        showarrow=False,
        font=dict(size=15, family="Arial, sans-serif"),
        xanchor="center",
        yanchor="middle"
    )
    
fig.update_layout(
    margin=dict(l=120, r=80, t=100, b=140)
)


fig.show()